In [1]:
# ==============================================================================
# NEXUS OptiML 2026 — Out-of-Time (OOT) Validated & GPU-Accelerated Pipeline
# Validation: 3-Fold Expanding Window Time-Series Split + Months 1-18 -> 19-24 OOT
# Models (AdaBoost Removed):
#   1. LightGBM (GPU / Fast Histogram, Optuna-Tuned on OOT Months 19-24)
#   2. CatBoost (Native GPU Pool with L2 Regularization)
#   3. TabM (GPU FP16 K=8 Parameter-Efficient Ensemble + Periodic Embeddings)
#   4. Random Forest (GPU cuML RF / Pre-Binned LightGBM RF)
#   5. Polynomial ElasticNet Logistic Regression (Degree-2 Poly + L1/L2 ElasticNet)
# ==============================================================================

import os
import re
import gc
import copy
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.preprocessing import OrdinalEncoder, PolynomialFeatures, StandardScaler, MinMaxScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import optuna

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

USE_GPU = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_GPU else "cpu")
T_START = time.time()

try:
    from cuml.ensemble import RandomForestClassifier as cuRF
    HAS_CUML = USE_GPU
except ImportError:
    HAS_CUML = False

def detect_lgb_device():
    """Probes LightGBM for CUDA/OpenCL GPU support and falls back cleanly to multi-thread CPU."""
    if not USE_GPU:
        return "cpu"
    X_tiny = np.random.randn(64, 8).astype(np.float32)
    y_tiny = np.random.randint(0, 2, size=64).astype(np.int32)
    for dev in ["cuda", "gpu"]:
        try:
            ds = lgb.Dataset(X_tiny, label=y_tiny, params={"verbose": -1})
            lgb.train({"objective": "binary", "device_type": dev, "verbose": -1}, ds, num_boost_round=1)
            return dev
        except Exception:
            pass
    return "cpu"

LGB_DEVICE = detect_lgb_device()
print(
    f"[{time.time() - T_START:5.1f}s] PyTorch Device: {DEVICE} | "
    f"LGB Device: {LGB_DEVICE} | cuML RF: {HAS_CUML}"
)

# ==============================================================================
# 1. FAST VECTORIZED UTILITIES (ZERO PYTHON LAMBDAS OR DATAFRAME MERGE COPIES)
# ==============================================================================
def fast_group_mode(df, group_cols, target_col):
    """Vectorized Cython group mode (0.3s on 1.88M rows)."""
    cols = group_cols if isinstance(group_cols, list) else [group_cols]
    sub = df[cols + [target_col]].dropna(subset=[target_col])
    counts = sub.groupby(cols + [target_col], sort=False).size().reset_index(name="_cnt")
    counts = counts.sort_values("_cnt", ascending=False).drop_duplicates(cols)
    if len(cols) == 1:
        return counts.set_index(cols[0])[target_col]
    return counts.set_index(cols)[target_col]

def map_table_inplace(df, ref_df, key_col, prefix=""):
    """Maps all columns from ref_df onto df in-place using Cython index.get_indexer (zero DataFrame copy)."""
    ref_indexed = ref_df.set_index(key_col)
    idx = ref_indexed.index.get_indexer(df[key_col])
    valid_mask = idx >= 0
    for col in ref_indexed.columns:
        out_col = f"{prefix}{col}" if col in df.columns else col
        vals = ref_indexed[col].values[idx]
        if not valid_mask.all() and np.issubdtype(vals.dtype, np.floating):
            vals = np.where(valid_mask, vals, np.nan)
        df[out_col] = vals

def fast_pair_count(code_a, code_b, shift_bits=20):
    """Computes exact group counts of (code_a, code_b) in 0.06s using bitwise int64 + bincount."""
    comp = (code_a.astype(np.int64) << shift_bits) | code_b.astype(np.int64)
    inv, _ = pd.factorize(comp, sort=False)
    counts = np.bincount(inv)
    return counts[inv].astype(np.int16), inv

def clean_region_name(val):
    if pd.isna(val):
        return "Unknown"
    s = re.sub(r"[^a-z0-9]", "", str(val).strip().lower())
    mapping = {
        "brazilsoutheast": "Brazil-Southeast", "brazilsutheast": "Brazil-Southeast",
        "brazilsouth": "Brazil-South",
        "brazilnortheast": "Brazil-Northeast", "brazilnrtheast": "Brazil-Northeast",
        "brazilcentralwest": "Brazil-CentralWest", "brazilcetralwest": "Brazil-CentralWest",
        "brazilnorth": "Brazil-North",
        "uknorth": "UK-North", "uknrth": "UK-North",
        "uksouth": "UK-South", "uksuth": "UK-South",
        "ukmidlands": "UK-Midlands", "ukmilands": "UK-Midlands",
        "uswest": "US-West",
        "useast": "US-East", "usest": "US-East", "usast": "US-East",
        "uscentral": "US-Central", "uscetral": "US-Central",
        "ussouth": "US-South", "ussuth": "US-South",
        "westerneurope": "Western Europe",
        "southerneurope": "Southern Europe", "southereurope": "Southern Europe",
        "nordics": "Nordics", "norics": "Nordics",
        "asiapacific": "Asia Pacific", "asiapcific": "Asia Pacific",
        "ireland": "Ireland", "ireand": "Ireland",
        "otherinternational": "Other International", "otherintrnational": "Other International",
        "americas": "Americas", "amercas": "Americas",
        "myanmaryangon": "Myanmar-Yangon",
        "myanmarmandalay": "Myanmar-Mandalay", "myanmarandalay": "Myanmar-Mandalay",
        "myanmarnaypyitaw": "Myanmar-Naypyitaw", "myanmaraypyitaw": "Myanmar-Naypyitaw",
    }
    return mapping.get(s, str(val).strip())

def clean_category_name(val):
    if pd.isna(val):
        return "unknown"
    s = re.sub(r"[^a-z0-9]", "", str(val).strip().lower())
    mapping = {
        "homedecor": "home_decor", "homeecor": "home_decor",
        "kitchendining": "kitchen_dining",
        "sportsleisure": "sports_leisure", "sportseisure": "sports_leisure",
        "toysgifts": "toys_gifts", "toysifts": "toys_gifts",
        "fashionaccessories": "fashion_accessories", "fashionacessories": "fashion_accessories",
        "officesupplies": "office_supplies", "officeupplies": "office_supplies",
        "healthbeauty": "health_beauty",
        "technology": "technology", "technlogy": "technology",
        "electronicsaccessories": "electronics_accessories",
        "gardendiy": "garden_diy", "gardediy": "garden_diy",
        "furniture": "furniture", "furnture": "furniture",
        "foodbeverages": "food_beverages", "foodbeerages": "food_beverages",
    }
    return mapping.get(s, str(val).strip().lower())

# ==============================================================================
# 2. DATA LOADING & SUPPLEMENTARY TABLE RELATIONAL PREPROCESSING
# ==============================================================================
DATA_DIR = "."  # Change to "/kaggle/input/nexus-opti-ml-2026-operation-insight" on Kaggle

stores_df = pd.read_csv(os.path.join(DATA_DIR, "stores.csv"))
calendar_df = pd.read_csv(os.path.join(DATA_DIR, "calendar.csv"))
customers_df = pd.read_csv(os.path.join(DATA_DIR, "customers.csv"))
products_df = pd.read_csv(os.path.join(DATA_DIR, "products.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sub_df = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))

print(f"[{time.time() - T_START:5.1f}s] Loaded train: {train_df.shape} | test: {test_df.shape}")

def prepare_reference_tables_fast(stores, customers, products, calendar):
    # --- 1. STORES ---
    reg_map_s = {k: clean_region_name(k) for k in stores["region"].unique()}
    stores["region"] = stores["region"].map(reg_map_s)
    stores["is_virtual_store"] = stores["size_sqm"].isna().astype(np.int8)
    stores["size_sqm"] = (
        stores["size_sqm"]
        .fillna(stores.groupby(["channel", "format"], sort=False)["size_sqm"].transform("median"))
        .fillna(0.0)
        .astype(np.float32)
    )
    stores["opened_date"] = pd.to_datetime(stores["opened_date"], cache=True)
    stores["store_vintage_pct"] = stores["opened_date"].rank(pct=True).astype(np.float32)
    stores["manager_tenure_years"] = stores["manager_tenure_years"].astype(np.float32)
    stores["seller_rating"] = stores["seller_rating"].astype(np.float32)
    stores.drop(columns=["channel"], inplace=True)

    # --- 2. CUSTOMERS ---
    reg_map_c = {k: clean_region_name(k) for k in customers["region"].unique()}
    customers["region_cust"] = customers["region"].map(reg_map_c)
    customers.drop(columns=["region"], inplace=True)
    customers["signup_date"] = pd.to_datetime(customers["signup_date"], cache=True)
    customers["record_updated_at"] = pd.to_datetime(customers["record_updated_at"], cache=True)

    tier_map = {"bronze": 1, "silver": 2, "gold": 3, "platinum": 4}
    customers["loyalty_rank"] = customers["loyalty_tier"].map(tier_map).fillna(1).astype(np.int8)
    customers = customers.sort_values(["customer_id", "record_updated_at"])
    customers["age_band"] = customers.groupby("customer_id", sort=False)["age_band"].transform("first")

    first_loyalty = customers.drop_duplicates("customer_id", keep="first").set_index("customer_id")["loyalty_rank"]
    cust_counts = customers["customer_id"].value_counts()
    customers = customers.drop_duplicates("customer_id", keep="last").copy()
    customers["cust_snapshot_count"] = customers["customer_id"].map(cust_counts).astype(np.int8)
    customers["loyalty_tier_delta"] = (
        customers["loyalty_rank"] - customers["customer_id"].map(first_loyalty)
    ).astype(np.int8)

    customers["age_band_was_null"] = customers["age_band"].isna().astype(np.int8)
    age_mode = fast_group_mode(customers, ["customer_type", "loyalty_tier"], "age_band")
    customers = customers.set_index(["customer_type", "loyalty_tier"])
    customers["age_band"] = customers["age_band"].fillna(age_mode)
    customers = customers.reset_index()
    customers["age_band"] = customers["age_band"].fillna("25-34")

    age_num_map = {"18-24": 21.0, "25-34": 30.0, "35-44": 40.0, "45-54": 50.0, "55-64": 60.0, "65+": 70.0}
    customers["age_midpoint"] = customers["age_band"].map(age_num_map).fillna(35.0).astype(np.float32)
    customers["newsletter_opt_in"] = customers["newsletter_opt_in"].astype(np.int8)

    # --- 3. PRODUCTS ---
    cat_map_p = {k: clean_category_name(k) for k in products["category"].unique()}
    products["category"] = products["category"].map(cat_map_p)
    products["launch_date"] = pd.to_datetime(products["launch_date"], cache=True)
    products["prod_launch_pct"] = products["launch_date"].rank(pct=True).astype(np.float32)

    products["volume_cm3"] = (
        products["length_cm"].clip(lower=1.0)
        * products["width_cm"].clip(lower=1.0)
        * products["height_cm"].clip(lower=1.0)
    ).astype(np.float32)
    products["max_dim_cm"] = products[["length_cm", "width_cm", "height_cm"]].max(axis=1).astype(np.float32)
    products["min_dim_cm"] = products[["length_cm", "width_cm", "height_cm"]].min(axis=1).astype(np.float32)
    products["aspect_ratio"] = (products["max_dim_cm"] / products["min_dim_cm"].clip(lower=1.0)).astype(np.float32)

    # Physical volumetric density imputation for missing weight_kg
    products["weight_was_null"] = (products["weight_kg"].isna() | (products["weight_kg"] <= 0)).astype(np.int8)
    valid_w = products["weight_kg"] > 0
    products["density"] = np.where(valid_w, (products["weight_kg"] * 1000.0) / products["volume_cm3"], np.nan)
    med_dens = (
        products.groupby(["category", "packaging_type"], sort=False)["density"]
        .transform("median")
        .fillna(products["density"].median())
    )
    products["weight_kg"] = np.where(
        valid_w, products["weight_kg"], (products["volume_cm3"] * med_dens) / 1000.0
    ).astype(np.float32)
    products["density"] = ((products["weight_kg"] * 1000.0) / products["volume_cm3"]).astype(np.float32)
    products["list_price"] = products["list_price"].astype(np.float32)
    products["list_price_per_kg"] = (products["list_price"] / products["weight_kg"].clip(lower=0.05)).astype(np.float32)

    cat_mean_p = products.groupby("category", sort=False)["list_price"].transform("mean")
    cat_med_p = products.groupby("category", sort=False)["list_price"].transform("median")
    cat_med_w = products.groupby("category", sort=False)["weight_kg"].transform("median")
    products["cat_mean_list_price"] = cat_mean_p.astype(np.float32)
    products["prod_price_vs_cat"] = (products["list_price"] / cat_med_p.clip(lower=0.1)).astype(np.float32)
    products["prod_weight_vs_cat"] = (products["weight_kg"] / cat_med_w.clip(lower=0.01)).astype(np.float32)

    # --- 4. CALENDAR (Post-Holiday Window + Economic Stress Metrics) ---
    calendar["week_start"] = pd.to_datetime(calendar["week_start"], cache=True)
    calendar = calendar.sort_values("week_start").reset_index(drop=True)

    # Post-holiday window: flag if order_ts falls in the 1-3 weeks following a major holiday
    h_shift1 = calendar["holiday_flag"].shift(1, fill_value=0).astype(np.int8)
    h_shift2 = calendar["holiday_flag"].shift(2, fill_value=0).astype(np.int8)
    h_shift3 = calendar["holiday_flag"].shift(3, fill_value=0).astype(np.int8)
    calendar["post_holiday_window"] = ((h_shift1 == 1) | (h_shift2 == 1) | (h_shift3 == 1)).astype(np.int8)
    calendar["post_holiday_intensity"] = (0.55 * h_shift1 + 0.30 * h_shift2 + 0.15 * h_shift3).astype(np.float32)

    # Economic stress metrics: include cpi and unemployment aligned to week_start + rolling changes
    calendar["cpi"] = calendar["cpi"].astype(np.float32)
    calendar["unemployment"] = calendar["unemployment"].astype(np.float32)
    calendar["fuel_price"] = calendar["fuel_price"].astype(np.float32)
    calendar["fuel_surcharge"] = calendar["fuel_surcharge"].astype(np.float32)
    calendar["temperature_f"] = calendar["temperature_f"].astype(np.float32)
    calendar["holiday_flag"] = calendar["holiday_flag"].astype(np.int8)

    calendar["cpi_pct_change_4w"] = calendar["cpi"].pct_change(4).fillna(0.0).astype(np.float32)
    calendar["unemp_diff_4w"] = calendar["unemployment"].diff(4).fillna(0.0).astype(np.float32)
    calendar["fuel_vs_4w_ma"] = (
        calendar["fuel_price"] / calendar["fuel_price"].rolling(4, min_periods=1).mean()
    ).astype(np.float32)
    # Composite economic stress index combining inflation momentum, fuel pressure, and unemployment
    cpi_z = (calendar["cpi"] - calendar["cpi"].mean()) / (calendar["cpi"].std() + 1e-5)
    unemp_z = (calendar["unemployment"] - calendar["unemployment"].mean()) / (calendar["unemployment"].std() + 1e-5)
    fuel_z = (calendar["fuel_price"] - calendar["fuel_price"].mean()) / (calendar["fuel_price"].std() + 1e-5)
    calendar["economic_stress_index"] = (0.4 * cpi_z + 0.3 * unemp_z + 0.3 * fuel_z).astype(np.float32)

    return stores, customers, products, calendar

stores_df, customers_df, products_df, calendar_df = prepare_reference_tables_fast(
    stores_df, customers_df, products_df, calendar_df
)
print(f"[{time.time() - T_START:5.1f}s] Reference tables cleaned & macro/post-holiday features engineered.")

# ==============================================================================
# 3. STRICTLY BACKWARD-IN-TIME HISTORICAL TARGET ENCODING (ZERO LEAKAGE!)
# ==============================================================================
def compute_causal_historical_encodings(df, n_train, y_train, tr_block, tr_week_idx):
    """
    Computes historical return rates strictly backwards in time (prior blocks/weeks only):
      - customer_historical_return_rate (prior orders only, smoothed toward category_return_rate)
      - product_historical_return_rate  (prior orders only, smoothed toward category_return_rate)
      - category_return_rate & store_return_rate & supplier_return_rate (prior orders only)
      - Fallback hierarchy for unseen entities: entity -> category_return_rate -> global_prior
    Runs in ~0.15s on 1.88M rows via C-level np.add.at accumulators!
    """
    n_total = len(df)
    global_default = float(y_train.mean())

    cat_all = df["_cat_code"].values
    cust_all = df["_cust_code"].values
    prod_all = df["_prod_code"].values
    store_all = df["_store_code"].values
    sup_all = df["_sup_code"].values

    cat_cnt = np.zeros(int(cat_all.max()) + 2, dtype=np.float64)
    cat_sum = np.zeros(int(cat_all.max()) + 2, dtype=np.float64)
    cust_cnt = np.zeros(int(cust_all.max()) + 2, dtype=np.float64)
    cust_sum = np.zeros(int(cust_all.max()) + 2, dtype=np.float64)
    prod_cnt = np.zeros(int(prod_all.max()) + 2, dtype=np.float64)
    prod_sum = np.zeros(int(prod_all.max()) + 2, dtype=np.float64)
    store_cnt = np.zeros(int(store_all.max()) + 2, dtype=np.float64)
    store_sum = np.zeros(int(store_all.max()) + 2, dtype=np.float64)
    sup_cnt = np.zeros(int(sup_all.max()) + 2, dtype=np.float64)
    sup_sum = np.zeros(int(sup_all.max()) + 2, dtype=np.float64)

    out_cat_rate = np.empty(n_total, dtype=np.float32)
    out_cust_rate = np.empty(n_total, dtype=np.float32)
    out_prod_rate = np.empty(n_total, dtype=np.float32)
    out_store_rate = np.empty(n_total, dtype=np.float32)
    out_sup_rate = np.empty(n_total, dtype=np.float32)
    out_cust_cnt = np.empty(n_total, dtype=np.float32)
    out_prod_cnt = np.empty(n_total, dtype=np.float32)

    cum_glob_cnt = 0.0
    cum_glob_sum = 0.0

    def assign_slice(idx):
        g_prior = (cum_glob_sum / cum_glob_cnt) if cum_glob_cnt >= 50 else global_default
        c_id = cat_all[idx]
        c_rate = (cat_sum[c_id] + 50.0 * g_prior) / (cat_cnt[c_id] + 50.0)
        out_cat_rate[idx] = c_rate.astype(np.float32)

        # Unseen customers (cu_c == 0) automatically fall back to c_rate (category_return_rate)!
        cu_id = cust_all[idx]
        cu_c = cust_cnt[cu_id]
        out_cust_rate[idx] = ((cust_sum[cu_id] + 10.0 * c_rate) / (cu_c + 10.0)).astype(np.float32)
        out_cust_cnt[idx] = np.log1p(cu_c).astype(np.float32)

        # Unseen products (pr_c == 0) automatically fall back to c_rate (category_return_rate)!
        pr_id = prod_all[idx]
        pr_c = prod_cnt[pr_id]
        out_prod_rate[idx] = ((prod_sum[pr_id] + 15.0 * c_rate) / (pr_c + 15.0)).astype(np.float32)
        out_prod_cnt[idx] = np.log1p(pr_c).astype(np.float32)

        st_id = store_all[idx]
        out_store_rate[idx] = ((store_sum[st_id] + 40.0 * c_rate) / (store_cnt[st_id] + 40.0)).astype(np.float32)

        sp_id = sup_all[idx]
        out_sup_rate[idx] = ((sup_sum[sp_id] + 25.0 * c_rate) / (sup_cnt[sp_id] + 25.0)).astype(np.float32)

    def update_slice(idx):
        nonlocal cum_glob_cnt, cum_glob_sum
        yb = y_train[idx].astype(np.float64)
        cum_glob_cnt += len(idx)
        cum_glob_sum += float(yb.sum())
        np.add.at(cat_cnt, cat_all[idx], 1.0)
        np.add.at(cat_sum, cat_all[idx], yb)
        np.add.at(cust_cnt, cust_all[idx], 1.0)
        np.add.at(cust_sum, cust_all[idx], yb)
        np.add.at(prod_cnt, prod_all[idx], 1.0)
        np.add.at(prod_sum, prod_all[idx], yb)
        np.add.at(store_cnt, store_all[idx], 1.0)
        np.add.at(store_sum, store_all[idx], yb)
        np.add.at(sup_cnt, sup_all[idx], 1.0)
        np.add.at(sup_sum, sup_all[idx], yb)

    # 1. Block 0 (Months 1-4): Expanding week-by-week (strictly prior weeks only)
    b0_indices = np.where(tr_block == 0)[0]
    b0_weeks = tr_week_idx[b0_indices]
    for wk in np.sort(np.unique(b0_weeks)):
        wk_idx = b0_indices[b0_weeks == wk]
        assign_slice(wk_idx)
        update_slice(wk_idx)

    # 2. Blocks 1..5 (Months 5-8, 9-12, 13-16, 17-20, 21-24): Frozen at end of prior block!
    for b in range(1, 6):
        b_idx = np.where(tr_block == b)[0]
        if len(b_idx) > 0:
            assign_slice(b_idx)
            update_slice(b_idx)

    # 3. Test Set (Months 25-30): Frozen at end of Month 24 (full training history)!
    test_idx = np.arange(n_train, n_total)
    assign_slice(test_idx)

    df["category_return_rate"] = out_cat_rate
    df["customer_historical_return_rate"] = out_cust_rate
    df["product_historical_return_rate"] = out_prod_rate
    df["store_return_rate"] = out_store_rate
    df["supplier_return_rate"] = out_sup_rate
    df["cust_prior_orders_log"] = out_cust_cnt
    df["prod_prior_orders_log"] = out_prod_cnt
    df["cust_vs_cat_return_rate"] = (out_cust_rate - out_cat_rate).astype(np.float32)
    df["prod_vs_cat_return_rate"] = (out_prod_rate - out_cat_rate).astype(np.float32)

# ==============================================================================
# 4. ZERO-COPY DOMAIN FEATURE ENGINEERING (BRACKETING, PRICING, LIFECYCLE, MACRO)
# ==============================================================================
def build_features_fast(train, test, stores, customers, products, calendar):
    n_train = len(train)
    y_train = train["returned"].astype(np.int8).values
    test_line_ids = test["order_line_id"].values

    train.drop(columns=["returned", "order_line_id"], inplace=True)
    test.drop(columns=["order_line_id"], inplace=True)
    train["is_test"] = np.int8(0)
    test["is_test"] = np.int8(1)

    df = pd.concat([train, test], axis=0, ignore_index=True)
    del train, test
    gc.collect()

    # Factorize primary entity keys into int32 codes ONCE
    df["_ord_code"] = pd.factorize(df["order_id"], sort=False)[0].astype(np.int32)
    df["_cust_code"] = pd.factorize(df["customer_id"], sort=False)[0].astype(np.int32)
    df["_prod_code"] = pd.factorize(df["product_id"], sort=False)[0].astype(np.int32)
    df["_store_code"] = pd.factorize(df["store_id"], sort=False)[0].astype(np.int32)

    # --- A. RELATIONAL NULL IMPUTATION ---
    ord_grp = df.groupby("_ord_code", sort=False)
    for col in ["tz_hint", "payment_method", "ship_mode", "session_clicks"]:
        df[f"{col}_was_null"] = df[col].isna().astype(np.int8)
        df[col] = ord_grp[col].transform("first")

    store_tz_map = fast_group_mode(df, "_store_code", "tz_hint")
    df["tz_hint"] = df["tz_hint"].fillna(df["_store_code"].map(store_tz_map)).fillna("UTC")

    df.loc[df["channel"] == "store", "ship_mode"] = df.loc[df["channel"] == "store", "ship_mode"].fillna("in_store")
    cust_sm_map = fast_group_mode(df, "_cust_code", "ship_mode")
    df["ship_mode"] = df["ship_mode"].fillna(df["_cust_code"].map(cust_sm_map)).fillna("standard")

    cust_pm_map = fast_group_mode(df, "_cust_code", "payment_method")
    df["payment_method"] = df["payment_method"].fillna(df["_cust_code"].map(cust_pm_map)).fillna("card")

    df["has_coupon"] = df["coupon_code"].notna().astype(np.int8)
    df["order_has_coupon"] = ord_grp["has_coupon"].transform("max").astype(np.int8)
    df.drop(columns=["coupon_code"], inplace=True)

    # --- B. ZERO-COPY IN-PLACE JOINS OF SUPPLEMENTARY TABLES ---
    map_table_inplace(df, products, "product_id")
    map_table_inplace(df, stores, "store_id")
    map_table_inplace(df, customers, "customer_id")

    df["order_dt"] = pd.to_datetime(df["order_ts"], cache=True)
    df["week_start"] = df["order_dt"].dt.floor("D") - pd.to_timedelta(df["order_dt"].dt.dayofweek, unit="D")
    min_wk, max_wk = calendar["week_start"].min(), calendar["week_start"].max()
    df["week_start"] = df["week_start"].clip(lower=min_wk, upper=max_wk)
    map_table_inplace(df, calendar, "week_start")

    df["_cat_code"] = pd.factorize(df["category"], sort=False)[0].astype(np.int32)
    df["_sup_code"] = pd.factorize(df["supplier_code"], sort=False)[0].astype(np.int32)
    df["_col_code"] = pd.factorize(df["color"], sort=False)[0].astype(np.int16)

    # --- C. OUTLIER REPAIR (100x CENTS BUG) & RELATIONAL DISCOUNT/FREIGHT IMPUTATION ---
    raw_line_spend = df["unit_price"] * df["quantity"].clip(lower=1)
    cents_bug = (
        (df["unit_price"] / df["list_price"].clip(lower=0.01) > 8.0)
        | ((df["order_value"] > 0) & ((df["order_value"] / raw_line_spend) < 0.05))
    )
    df["is_cents_bug"] = cents_bug.astype(np.int8)
    df["clean_unit_price"] = np.where(cents_bug, df["unit_price"] / 100.0, df["unit_price"]).astype(np.float32)

    df["discount_was_null"] = df["discount"].isna().astype(np.int8)
    implied_disc = np.clip(
        1.0 - (df["clean_unit_price"] / (0.963 * df["list_price"].clip(lower=0.01))), 0.0, 0.80
    ).astype(np.float32)
    df["discount"] = df["discount"].fillna(implied_disc).astype(np.float32)

    sc_med = df.groupby(["device_type", "channel"], sort=False)["session_clicks"].transform("median")
    df["session_clicks"] = df["session_clicks"].fillna(sc_med).fillna(9.0).astype(np.float32)

    df["freight_was_null"] = df["freight_cost"].isna().astype(np.int8)
    df["line_weight_kg"] = (df["weight_kg"] * df["quantity"]).astype(np.float32)
    df["line_volume_cm3"] = (df["volume_cm3"] * df["quantity"]).astype(np.float32)
    w_bin = pd.qcut(df["line_weight_kg"].rank(method="first"), q=10, labels=False)
    fr_med = df.groupby(["ship_mode", "channel", w_bin], sort=False)["freight_cost"].transform("median")
    df["freight_cost"] = df["freight_cost"].fillna(fr_med).fillna(df["freight_cost"].median()).astype(np.float32)

    # --- D. HIGH-IMPACT SIGNAL 1: "BRACKETING" & ORDER COMPOSITION (0.2s VECTORIZED) ---
    same_cat_cnt, oc_inv = fast_pair_count(df["_ord_code"].values, df["_cat_code"].values, 16)
    same_brand_cnt, _ = fast_pair_count(df["_ord_code"].values, df["_sup_code"].values, 16)
    same_prod_cnt, _ = fast_pair_count(df["_ord_code"].values, df["_prod_code"].values, 20)

    df["same_category_count_in_order"] = same_cat_cnt
    df["same_brand_count_in_order"] = same_brand_cnt
    df["same_product_count_in_order"] = same_prod_cnt
    df["is_category_bracketing"] = (same_cat_cnt > 1).astype(np.int8)
    df["is_brand_bracketing"] = (same_brand_cnt > 1).astype(np.int8)

    # Detect multi-color bracketing inside the same (order_id, category)
    n_oc = int(oc_inv.max()) + 1
    min_col = np.full(n_oc, 32767, dtype=np.int16)
    max_col = np.full(n_oc, -1, dtype=np.int16)
    col_arr = df["_col_code"].values
    np.minimum.at(min_col, oc_inv, col_arr)
    np.maximum.at(max_col, oc_inv, col_arr)
    df["is_color_bracketing"] = (min_col[oc_inv] != max_col[oc_inv]).astype(np.int8)

    df["expected_line_value"] = (df["clean_unit_price"] * df["quantity"]).astype(np.float32)
    df["item_order_share"] = (df["expected_line_value"] / df["order_value"].clip(lower=0.01)).astype(np.float32)
    df["order_size_ratio"] = (df["quantity"] / df["items_in_order"].clip(lower=1)).astype(np.float32)

    # --- E. HIGH-IMPACT SIGNAL 2: PRICING & PERCEIVED VALUE MISMATCH ---
    df["discount_pct"] = (
        (df["list_price"] - df["clean_unit_price"]) / df["list_price"].clip(lower=0.01)
    ).astype(np.float32)
    df["discount_residual"] = (df["discount"] - df["discount_pct"]).astype(np.float32)
    cat_mean_unit = df.groupby("_cat_code", sort=False)["clean_unit_price"].transform("mean").astype(np.float32)
    df["price_deviation_from_category"] = (df["clean_unit_price"] - cat_mean_unit).astype(np.float32)
    df["price_ratio_to_category"] = (df["clean_unit_price"] / cat_mean_unit.clip(lower=0.01)).astype(np.float32)
    df["freight_ratio"] = (df["freight_cost"] / (df["order_value"] + 1.0)).astype(np.float32)
    df["is_free_or_low_freight"] = (df["freight_ratio"] < 0.03).astype(np.int8)
    df["freight_per_kg"] = (df["freight_cost"] / (df["line_weight_kg"] + 0.1)).astype(np.float32)
    df["dollar_discount_total"] = (
        (df["list_price"] - df["clean_unit_price"]).clip(lower=0.0) * df["quantity"]
    ).astype(np.float32)

    # Order-level aggregations
    ord_grp = df.groupby("_ord_code", sort=False)
    df["obs_order_lines"] = ord_grp["quantity"].transform("count").astype(np.int16)
    df["ord_sum_val"] = ord_grp["order_value"].transform("sum").astype(np.float32)
    df["ord_mean_val"] = ord_grp["order_value"].transform("mean").astype(np.float32)
    df["ord_max_val"] = ord_grp["order_value"].transform("max").astype(np.float32)
    df["ord_sum_qty"] = ord_grp["quantity"].transform("sum").astype(np.int32)
    df["ord_sum_weight"] = ord_grp["line_weight_kg"].transform("sum").astype(np.float32)
    df["ord_mean_disc"] = ord_grp["discount"].transform("mean").astype(np.float32)
    df["is_multi_store_order"] = (
        ord_grp["_store_code"].transform("min") != ord_grp["_store_code"].transform("max")
    ).astype(np.int8)
    df["is_multi_cat_order"] = (
        ord_grp["_cat_code"].transform("min") != ord_grp["_cat_code"].transform("max")
    ).astype(np.int8)
    df["same_cat_share_of_order_lines"] = (
        df["same_category_count_in_order"] / df["obs_order_lines"].clip(lower=1)
    ).astype(np.float32)
    df["line_val_share_of_order"] = (df["order_value"] / df["ord_sum_val"].clip(lower=0.01)).astype(np.float32)
    df["line_no_rel_pos"] = (df["line_no"] / df["items_in_order"].clip(lower=1)).astype(np.float32)
    df["obs_to_reported_lines_ratio"] = (df["obs_order_lines"] / df["items_in_order"].clip(lower=1)).astype(np.float32)
    df["clicks_per_line"] = (df["session_clicks"] / df["obs_order_lines"].clip(lower=1)).astype(np.float32)

    # --- F. HIGH-IMPACT SIGNAL 4: TEMPORAL & LIFECYCLE GAPS ---
    df["customer_tenure_days"] = (
        (df["order_dt"] - df["signup_date"]).dt.days.clip(lower=-30).fillna(180).astype(np.float32)
    )
    df["product_age_days"] = (
        (df["order_dt"] - df["launch_date"]).dt.days.clip(lower=-30).fillna(180).astype(np.float32)
    )
    df["store_age_days"] = (
        (df["order_dt"] - df["opened_date"]).dt.days.clip(lower=0).fillna(1000).astype(np.float32)
    )
    df["log_customer_tenure_days"] = np.log1p(df["customer_tenure_days"].clip(lower=0)).astype(np.float32)
    df["log_product_age_days"] = np.log1p(df["product_age_days"].clip(lower=0)).astype(np.float32)
    df["log_store_age_days"] = np.log1p(df["store_age_days"].clip(lower=0)).astype(np.float32)
    df["is_new_customer_30d"] = (df["customer_tenure_days"] <= 30).astype(np.int8)
    df["is_new_product_60d"] = (df["product_age_days"] <= 60).astype(np.int8)
    df["updated_after_order"] = (df["record_updated_at"] > df["order_dt"]).astype(np.int8)

    # Cross-table geographic alignment & cyclical seasonality
    df["same_region_cust_store"] = (df["region_cust"] == df["region"]).astype(np.int8)
    df["cust_country"] = df["region_cust"].str.split("-").str[0]
    df["store_country"] = df["region"].str.split("-").str[0]
    df["same_country_cust_store"] = (df["cust_country"] == df["store_country"]).astype(np.int8)

    df["order_hour"] = df["order_dt"].dt.hour.astype(np.int8)
    df["order_dow"] = df["order_dt"].dt.dayofweek.astype(np.int8)
    df["order_month"] = df["order_dt"].dt.month.astype(np.int8)
    df["is_weekend"] = (df["order_dow"] >= 5).astype(np.int8)
    df["hour_sin"] = np.sin(2.0 * np.pi * df["order_hour"] / 24.0).astype(np.float32)
    df["hour_cos"] = np.cos(2.0 * np.pi * df["order_hour"] / 24.0).astype(np.float32)
    df["dow_sin"] = np.sin(2.0 * np.pi * df["order_dow"] / 7.0).astype(np.float32)
    df["dow_cos"] = np.cos(2.0 * np.pi * df["order_dow"] / 7.0).astype(np.float32)
    df["month_sin"] = np.sin(2.0 * np.pi * df["order_month"] / 12.0).astype(np.float32)
    df["month_cos"] = np.cos(2.0 * np.pi * df["order_month"] / 12.0).astype(np.float32)

    # Target-free customer, product & store behavioral profiles
    cust_grp = df.groupby("_cust_code", sort=False)
    df["cust_mean_val"] = cust_grp["order_value"].transform("mean").astype(np.float32)
    df["cust_mean_disc"] = cust_grp["discount"].transform("mean").astype(np.float32)
    df["cust_mean_qty"] = cust_grp["quantity"].transform("mean").astype(np.float32)
    df["cust_gift_rate"] = cust_grp["gift_wrap"].transform("mean").astype(np.float32)
    df["cust_mean_ret_win"] = cust_grp["return_window_days"].transform("mean").astype(np.float32)
    df["val_vs_cust_mean"] = (df["order_value"] / df["cust_mean_val"].clip(lower=0.1)).astype(np.float32)
    df["disc_vs_cust_mean"] = (df["discount"] - df["cust_mean_disc"]).astype(np.float32)

    prod_grp = df.groupby("_prod_code", sort=False)
    df["prod_mean_disc"] = prod_grp["discount"].transform("mean").astype(np.float32)
    df["prod_mean_qty"] = prod_grp["quantity"].transform("mean").astype(np.float32)
    df["prod_gift_rate"] = prod_grp["gift_wrap"].transform("mean").astype(np.float32)
    df["prod_mean_unit_price"] = prod_grp["clean_unit_price"].transform("mean").astype(np.float32)
    df["unit_price_vs_prod_mean"] = (
        df["clean_unit_price"] / df["prod_mean_unit_price"].clip(lower=0.01)
    ).astype(np.float32)

    store_grp = df.groupby("_store_code", sort=False)
    df["store_mean_val"] = store_grp["order_value"].transform("mean").astype(np.float32)
    df["store_mean_disc"] = store_grp["discount"].transform("mean").astype(np.float32)
    df["store_mean_freight"] = store_grp["freight_cost"].transform("mean").astype(np.float32)

    # --- G. DEFINE CHRONOLOGICAL BLOCKS & CAUSAL HISTORICAL RETURN RATES ---
    train_ts = df.loc[:n_train - 1, "order_dt"].reset_index(drop=True)
    ts_rank = train_ts.rank(pct=True).values.astype(np.float32)
    # 6 equal 4-month chronological blocks across Months 1-24:
    # Block 0: M1-4 | Block 1: M5-8 | Block 2: M9-12 | Block 3: M13-16 | Block 4: M17-20 | Block 5: M21-24
    tr_block = np.minimum(5, (ts_rank * 6.0).astype(np.int32))
    tr_week_idx = ((train_ts - train_ts.min()).dt.days // 7).values.astype(np.int32)

    compute_causal_historical_encodings(df, n_train, y_train, tr_block, tr_week_idx)

    drop_cols = [
        "order_id", "customer_id", "product_id", "unit_price", "order_ts", "order_dt",
        "week_start", "signup_date", "record_updated_at", "launch_date", "opened_date",
        "_ord_code", "_cust_code", "_prod_code", "_store_code", "_cat_code", "_sup_code", "_col_code"
    ]
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

    train_out = df.iloc[:n_train].drop(columns=["is_test"]).reset_index(drop=True)
    test_out = df.iloc[n_train:].drop(columns=["is_test"]).reset_index(drop=True)
    del df
    gc.collect()

    return train_out, test_out, y_train, tr_block, ts_rank, test_line_ids

train_fe, test_fe, y_all, tr_block_all, tr_ts_rank, test_line_ids = build_features_fast(
    train_df, test_df, stores_df, customers_df, products_df, calendar_df
)
del stores_df, customers_df, products_df, calendar_df
gc.collect()
print(f"[{time.time() - T_START:5.1f}s] Feature engineering complete: {train_fe.shape[1]} total features.")

# ==============================================================================
# 5. EXPANDING WINDOW OOT SPLITS, OUTLIER WINSORIZATION, STANDARDIZATION & NORMALIZATION
# ==============================================================================
CAT_COLS = [
    "store_id", "supplier_code", "category", "packaging_type", "color",
    "tz_hint", "channel", "payment_method", "ship_mode", "promo_channel_code",
    "warehouse_zone", "picker_shift", "device_type", "region", "format",
    "region_cust", "customer_type", "age_band", "preferred_contact",
    "loyalty_tier", "cust_country", "store_country", "banner_id"
]
CAT_COLS = [c for c in CAT_COLS if c in train_fe.columns]
NUM_COLS = [c for c in train_fe.columns if c not in CAT_COLS]

def build_temporal_splits_and_matrices(train, test, tr_block, ts_rank, cat_cols, num_cols):
    # 1. Construct 3-Fold Expanding Window Time-Series Splits:
    #    Fold 1: Train Months 1-12 (Blocks 0..2) -> Validate Months 13-16 (Block 3)
    #    Fold 2: Train Months 1-16 (Blocks 0..3) -> Validate Months 17-20 (Block 4)
    #    Fold 3: Train Months 1-20 (Blocks 0..4) -> Validate Months 21-24 (Block 5)
    expanding_folds = [
        (np.where(tr_block <= 2)[0], np.where(tr_block == 3)[0], "Months 13-16"),
        (np.where(tr_block <= 3)[0], np.where(tr_block == 4)[0], "Months 17-20"),
        (np.where(tr_block <= 4)[0], np.where(tr_block == 5)[0], "Months 21-24"),
    ]
    # Primary Out-of-Time (OOT) Split: Train Months 1-18 (ts_rank <= 0.75) -> Validate Months 19-24 (ts_rank > 0.75)
    oot_tr_mask = ts_rank <= 0.75
    oot_va_mask = ts_rank > 0.75

    # 2. Ordinal encode categoricals (shifted +1 so unknown -1 becomes index 0)
    ord_enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, dtype=np.int32)
    train[cat_cols] = ord_enc.fit_transform(train[cat_cols].astype(str)) + 1
    test[cat_cols] = ord_enc.transform(test[cat_cols].astype(str)) + 1

    # 3. Winsorize numerical outliers at [0.5%, 99.5%] based strictly on training percentiles
    for col in num_cols:
        tr_vals = train[col].values.astype(np.float32)
        te_vals = test[col].values.astype(np.float32)
        tr_vals = np.where(np.isfinite(tr_vals), tr_vals, np.nan)
        te_vals = np.where(np.isfinite(te_vals), te_vals, np.nan)
        med = float(np.nanmedian(tr_vals))
        tr_vals = np.nan_to_num(tr_vals, nan=med)
        te_vals = np.nan_to_num(te_vals, nan=med)
        q_low, q_high = np.percentile(tr_vals, [0.5, 99.5])
        if q_high > q_low:
            np.clip(tr_vals, q_low, q_high, out=tr_vals)
            np.clip(te_vals, q_low, q_high, out=te_vals)
        train[col] = tr_vals
        test[col] = te_vals

    # 4. Contiguous Float32 Matrices for Tree Models (LightGBM, CatBoost, Random Forest)
    feature_names = num_cols + cat_cols
    cat_indices = list(range(len(num_cols), len(feature_names)))
    X_tr_tree = np.ascontiguousarray(train[feature_names].values, dtype=np.float32)
    X_te_tree = np.ascontiguousarray(test[feature_names].values, dtype=np.float32)

    # 5. Power Transform (Log-Modulus) + Standardization (Z-Score) + Normalization (MinMax [-1, 1]) for TabM & Poly-LR
    X_tr_num = np.ascontiguousarray(train[num_cols].values, dtype=np.float32)
    X_te_num = np.ascontiguousarray(test[num_cols].values, dtype=np.float32)

    tr_mean_raw = X_tr_num.mean(axis=0, keepdims=True)
    tr_std_raw = X_tr_num.std(axis=0, keepdims=True) + 1e-6
    skews = np.abs(((X_tr_num - tr_mean_raw) / tr_std_raw) ** 3).mean(axis=0)
    skew_mask = skews > 1.8
    if skew_mask.any():
        X_tr_num[:, skew_mask] = np.sign(X_tr_num[:, skew_mask]) * np.log1p(np.abs(X_tr_num[:, skew_mask]))
        X_te_num[:, skew_mask] = np.sign(X_te_num[:, skew_mask]) * np.log1p(np.abs(X_te_num[:, skew_mask]))

    # StandardScaler (Zero mean, unit variance)
    std_scaler = StandardScaler()
    X_tr_std = std_scaler.fit_transform(X_tr_num).astype(np.float32)
    X_te_std = np.clip(std_scaler.transform(X_te_num), -10.0, 10.0).astype(np.float32)

    # MinMaxScaler (Bounded [-1, 1] normalization blended with Z-score for smooth gradient stability)
    mm_scaler = MinMaxScaler(feature_range=(-1.0, 1.0))
    X_tr_norm = mm_scaler.fit_transform(X_tr_std).astype(np.float32)
    X_te_norm = np.clip(mm_scaler.transform(X_te_std), -1.5, 1.5).astype(np.float32)

    X_tr_num_dl = np.ascontiguousarray(0.7 * X_tr_std + 0.3 * X_tr_norm, dtype=np.float32)
    X_te_num_dl = np.ascontiguousarray(0.7 * X_te_std + 0.3 * X_te_norm, dtype=np.float32)
    del X_tr_num, X_te_num, X_tr_std, X_te_std, X_tr_norm, X_te_norm

    X_tr_cat_dl = np.ascontiguousarray(train[cat_cols].values, dtype=np.int32)
    X_te_cat_dl = np.ascontiguousarray(test[cat_cols].values, dtype=np.int32)
    cat_cardinalities = [
        int(max(X_tr_cat_dl[:, i].max(), X_te_cat_dl[:, i].max())) + 1 for i in range(len(cat_cols))
    ]

    return (
        X_tr_tree, X_te_tree, cat_indices,
        X_tr_num_dl, X_te_num_dl,
        X_tr_cat_dl, X_te_cat_dl,
        cat_cardinalities, expanding_folds, oot_tr_mask, oot_va_mask
    )

(
    X_all, X_test, CAT_INDICES,
    X_tr_num_dl, X_te_num_dl,
    X_tr_cat_dl, X_te_cat_dl,
    CAT_CARDS, EXPANDING_FOLDS, oot_tr_mask, oot_va_mask
) = build_temporal_splits_and_matrices(
    train_fe, test_fe, tr_block_all, tr_ts_rank, CAT_COLS, NUM_COLS
)
del train_fe, test_fe
gc.collect()
print(f"[{time.time() - T_START:5.1f}s] Matrices ready | Tree: {X_all.shape} | TabM/Poly Num: {X_tr_num_dl.shape}")

# ==============================================================================
# 6A. TABULAR DEEP LEARNING: GPU TabM (PARAMETER-EFFICIENT ENSEMBLE, FP16 VRAM)
# ==============================================================================
class LinearEfficientEnsemble(nn.Module):
    """
    Core TabM Layer (Gorishniy et al.):
    Computes K parallel ensemble pathways sharing a single base weight matrix W
    modulated by rank-1 adapters R (K, in_dim), S (K, out_dim), and biases B (K, out_dim).
    """
    def __init__(
        self,
        in_features: int,
        out_features: int,
        k: int = 8,
        random_sign_init: bool = False,
        scale_init: float = 0.05,
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.k = k

        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.r = nn.Parameter(torch.empty(k, in_features))
        self.s = nn.Parameter(torch.empty(k, out_features))
        self.bias = nn.Parameter(torch.empty(k, out_features))
        self.reset_parameters(random_sign_init, scale_init)

    def reset_parameters(self, random_sign_init: bool, scale_init: float):
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        with torch.no_grad():
            if random_sign_init:
                r_signs = torch.randint(0, 2, self.r.shape, dtype=self.r.dtype) * 2.0 - 1.0
                s_signs = torch.randint(0, 2, self.s.shape, dtype=self.s.dtype) * 2.0 - 1.0
                self.r.copy_(r_signs * (1.0 + torch.randn_like(self.r) * scale_init))
                self.s.copy_(s_signs * (1.0 + torch.randn_like(self.s) * scale_init))
            else:
                self.r.copy_(1.0 + torch.randn_like(self.r) * scale_init)
                self.s.copy_(1.0 + torch.randn_like(self.s) * scale_init)
            bound = 1.0 / np.sqrt(self.in_features)
            nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_scaled = x * self.r.unsqueeze(0)
        out = F.linear(x_scaled, self.weight)
        return out * self.s.unsqueeze(0) + self.bias.unsqueeze(0)


class TabMClassifier(nn.Module):
    """
    TabM Architecture for Tabular Classification:
    - Periodic + Linear Numerical Embeddings (PLR-Lite) + Categorical Entity Embeddings
    - K=8 Parameter-Efficient Ensemble Pathways (LinearEfficientEnsemble)
    - Pre-Norm Residual Blocks with SiLU activation and Dropout
    - K Independent Classification Heads trained jointly on per-member BCE loss
    """
    def __init__(
        self,
        num_features: int,
        cat_cardinalities: list,
        k: int = 8,
        hidden_dim: int = 256,
        n_blocks: int = 2,
        emb_drop: float = 0.15,
        drop_rate: float = 0.25,
    ):
        super().__init__()
        self.k = k
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=card + 1, embedding_dim=min(24, max(4, (card + 1) // 2)))
            for card in cat_cardinalities
        ])
        total_emb_dim = sum(e.embedding_dim for e in self.embeddings)

        self.num_norm = nn.BatchNorm1d(num_features)
        self.num_freq = nn.Parameter(torch.randn(num_features) * 0.1)
        self.num_phase = nn.Parameter(torch.zeros(num_features))
        self.num_gate = nn.Linear(num_features * 2, num_features)

        in_dim = num_features + total_emb_dim
        self.emb_dropout = nn.Dropout(emb_drop)

        self.in_ensemble = LinearEfficientEnsemble(
            in_dim, hidden_dim, k=k, random_sign_init=True, scale_init=0.05
        )
        self.in_act = nn.Sequential(nn.LayerNorm(hidden_dim), nn.SiLU())

        self.blocks = nn.ModuleList([
            nn.ModuleDict({
                "norm": nn.LayerNorm(hidden_dim),
                "lin1": LinearEfficientEnsemble(hidden_dim, hidden_dim, k=k, random_sign_init=False),
                "act": nn.SiLU(),
                "drop": nn.Dropout(drop_rate),
                "lin2": LinearEfficientEnsemble(hidden_dim, hidden_dim, k=k, random_sign_init=False),
            })
            for _ in range(n_blocks)
        ])

        self.out_norm = nn.LayerNorm(hidden_dim)
        self.head_weight = nn.Parameter(torch.empty(k, hidden_dim))
        self.head_bias = nn.Parameter(torch.zeros(k))
        nn.init.kaiming_uniform_(self.head_weight, a=np.sqrt(5))

    def forward(self, x_num: torch.Tensor, x_cat: torch.Tensor) -> torch.Tensor:
        xn = self.num_norm(x_num)
        xn_periodic = torch.sin(xn * self.num_freq.unsqueeze(0) + self.num_phase.unsqueeze(0))
        xn = xn * torch.sigmoid(self.num_gate(torch.cat([xn, xn_periodic], dim=1)))

        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        xe = self.emb_dropout(torch.cat(embs, dim=1))
        x0 = torch.cat([xn, xe], dim=1)

        h = self.in_act(self.in_ensemble(x0.unsqueeze(1).expand(-1, self.k, -1)))
        for blk in self.blocks:
            res = blk["norm"](h)
            res = blk["lin1"](res)
            res = blk["act"](res)
            res = blk["drop"](res)
            res = blk["lin2"](res)
            h = h + res

        h = self.out_norm(h)
        logits_k = (h * self.head_weight.unsqueeze(0)).sum(dim=-1) + self.head_bias.unsqueeze(0)
        return logits_k


def train_tabm_fold_vram(
    X_num_all_gpu, X_cat_all_gpu, y_all_gpu, w_all_gpu,
    X_num_test_gpu, X_cat_test_gpu,
    tr_idx, val_idx, y_val_cpu, cat_cards, fold,
    k=8, epochs=12, batch_size=8192, max_lr=2.5e-3
):
    torch.manual_seed(SEED + fold)
    model = TabMClassifier(
        num_features=X_num_all_gpu.shape[1],
        cat_cardinalities=cat_cards,
        k=k,
        hidden_dim=256,
        n_blocks=2,
        emb_drop=0.15,
        drop_rate=0.25,
    ).to(DEVICE)

    decay_params, no_decay_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if any(nd in name for nd in [".r", ".s", "bias", "norm", "num_freq", "num_phase"]):
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    tr_idx_gpu = torch.from_numpy(tr_idx).long().to(DEVICE)
    val_idx_gpu = torch.from_numpy(val_idx).long().to(DEVICE)
    n_tr = len(tr_idx_gpu)
    steps_per_epoch = (n_tr + batch_size - 1) // batch_size

    optimizer = torch.optim.AdamW(
        [
            {"params": decay_params, "weight_decay": 1e-4},
            {"params": no_decay_params, "weight_decay": 0.0},
        ],
        lr=max_lr / 10.0,
    )
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=max_lr, epochs=epochs, steps_per_epoch=steps_per_epoch, pct_start=0.25
    )
    scaler = torch.cuda.amp.GradScaler(enabled=USE_GPU)

    best_auc = -1.0
    best_val_preds = None
    best_weights = None

    for epoch in range(epochs):
        model.train()
        perm = tr_idx_gpu[torch.randperm(n_tr, device=DEVICE)]
        for i in range(0, n_tr, batch_size):
            b_idx = perm[i : i + batch_size]
            if len(b_idx) < 16:
                continue
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_GPU, dtype=torch.float16):
                logits_k = model(X_num_all_gpu[b_idx], X_cat_all_gpu[b_idx].long())
                y_k = y_all_gpu[b_idx].unsqueeze(1).expand_as(logits_k)
                w_k = w_all_gpu[b_idx].unsqueeze(1).expand_as(logits_k)
                loss = (F.binary_cross_entropy_with_logits(logits_k, y_k, reduction="none") * w_k).mean()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        model.eval()
        val_preds_list = []
        with torch.no_grad():
            for i in range(0, len(val_idx_gpu), 16384):
                b_idx = val_idx_gpu[i : i + 16384]
                with torch.cuda.amp.autocast(enabled=USE_GPU, dtype=torch.float16):
                    logits_k = model(X_num_all_gpu[b_idx], X_cat_all_gpu[b_idx].long())
                    probs = torch.sigmoid(logits_k).mean(dim=1)
                val_preds_list.append(probs.float().cpu().numpy())

        val_preds = np.concatenate(val_preds_list)
        val_auc = roc_auc_score(y_val_cpu, val_preds)
        if val_auc > best_auc:
            best_auc = val_auc
            best_val_preds = val_preds
            best_weights = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_weights)
    model.eval()
    test_preds_list = []
    n_te = len(X_num_test_gpu)
    with torch.no_grad():
        for i in range(0, n_te, 16384):
            with torch.cuda.amp.autocast(enabled=USE_GPU, dtype=torch.float16):
                logits_k = model(
                    X_num_test_gpu[i : i + 16384], X_cat_test_gpu[i : i + 16384].long()
                )
                probs = torch.sigmoid(logits_k).mean(dim=1)
            test_preds_list.append(probs.float().cpu().numpy())

    test_preds = np.concatenate(test_preds_list)
    del model, optimizer, scheduler, tr_idx_gpu, val_idx_gpu, best_weights
    if USE_GPU:
        torch.cuda.empty_cache()
    return best_val_preds, test_preds

# ==============================================================================
# 6B. POLYNOMIAL FEATURES + ELASTICNET (L1 + L2) LOGISTIC REGRESSION (GPU + SGD)
# ==============================================================================
class GPUPolyElasticNetLogistic(nn.Module):
    """
    Degree-2 Polynomial Logistic Regression trained on GPU with explicit
    ElasticNet (L1 Lasso + L2 Ridge) regularization to prevent overfitting.
    """
    def __init__(self, in_dim, top_idx_gpu, qi_gpu, qj_gpu):
        super().__init__()
        self.top_idx = top_idx_gpu
        self.qi = qi_gpu
        self.qj = qj_gpu
        self.norm = nn.BatchNorm1d(in_dim)
        self.linear = nn.Linear(in_dim, 1, bias=True)

    def forward(self, x_num):
        x_top = x_num[:, self.top_idx]
        x_quad = x_top[:, self.qi] * x_top[:, self.qj]
        x_poly = self.norm(torch.cat([x_num, x_quad], dim=1))
        return self.linear(x_poly).squeeze(-1)

    def elasticnet_penalty(self, alpha=1e-3, l1_ratio=0.5):
        w_param = self.linear.weight
        l1_pen = torch.abs(w_param).sum()
        l2_pen = 0.5 * (w_param ** 2).sum()
        return alpha * (l1_ratio * l1_pen + (1.0 - l1_ratio) * l2_pen)

def train_poly_elasticnet_fold(
    X_num_all_gpu, y_all_gpu, w_all_gpu, X_num_test_gpu,
    X_num_cpu, y_cpu, w_cpu, X_test_num_cpu,
    tr_idx, val_idx, top_poly_idx, q_i, q_j, poly_transformer, fold,
    alpha=1e-3, l1_ratio=0.5, epochs=8, batch_size=16384
):
    torch.manual_seed(SEED + 100 + fold)
    n_num = X_num_all_gpu.shape[1]
    in_dim = n_num + len(q_i)

    model_plr = GPUPolyElasticNetLogistic(
        in_dim=in_dim,
        top_idx_gpu=torch.from_numpy(top_poly_idx).long().to(DEVICE),
        qi_gpu=torch.from_numpy(q_i).long().to(DEVICE),
        qj_gpu=torch.from_numpy(q_j).long().to(DEVICE),
    ).to(DEVICE)

    tr_idx_gpu = torch.from_numpy(tr_idx).long().to(DEVICE)
    val_idx_gpu = torch.from_numpy(val_idx).long().to(DEVICE)
    n_tr = len(tr_idx_gpu)

    optimizer = torch.optim.AdamW(model_plr.parameters(), lr=5e-3, weight_decay=0.0)
    criterion = nn.BCEWithLogitsLoss(reduction="none")

    best_auc = -1.0
    best_gpu_val = None
    best_state = None

    for epoch in range(epochs):
        model_plr.train()
        perm = tr_idx_gpu[torch.randperm(n_tr, device=DEVICE)]
        for i in range(0, n_tr, batch_size):
            b_idx = perm[i : i + batch_size]
            if len(b_idx) < 16:
                continue
            optimizer.zero_grad(set_to_none=True)
            logits = model_plr(X_num_all_gpu[b_idx])
            bce_loss = (criterion(logits, y_all_gpu[b_idx]) * w_all_gpu[b_idx]).mean()
            loss = bce_loss + model_plr.elasticnet_penalty(alpha=alpha, l1_ratio=l1_ratio)
            loss.backward()
            optimizer.step()

        model_plr.eval()
        val_chunks = []
        with torch.no_grad():
            for i in range(0, len(val_idx_gpu), 32768):
                b_idx = val_idx_gpu[i : i + 32768]
                val_chunks.append(torch.sigmoid(model_plr(X_num_all_gpu[b_idx])).cpu().numpy())
        curr_val = np.concatenate(val_chunks)
        curr_auc = roc_auc_score(y_cpu[val_idx], curr_val)
        if curr_auc > best_auc:
            best_auc = curr_auc
            best_gpu_val = curr_val
            best_state = copy.deepcopy(model_plr.state_dict())

    model_plr.load_state_dict(best_state)
    model_plr.eval()
    test_chunks = []
    with torch.no_grad():
        for i in range(0, len(X_num_test_gpu), 32768):
            test_chunks.append(torch.sigmoid(model_plr(X_num_test_gpu[i : i + 32768])).cpu().numpy())
    gpu_test_preds = np.concatenate(test_chunks)

    del model_plr, optimizer, tr_idx_gpu, val_idx_gpu, best_state
    if USE_GPU:
        torch.cuda.empty_cache()

    # Scikit-Learn PolynomialFeatures(degree=2) + ElasticNet SGDClassifier
    rng = np.random.RandomState(SEED + fold)
    sub_idx = rng.choice(tr_idx, size=min(100_000, len(tr_idx)), replace=False)
    rem_idx = np.setdiff1d(np.arange(n_num), top_poly_idx)

    X_tr_poly = np.hstack([
        poly_transformer.transform(X_num_cpu[sub_idx][:, top_poly_idx]),
        X_num_cpu[sub_idx][:, rem_idx]
    ])
    sgd_lr = SGDClassifier(
        loss="log_loss",
        penalty="elasticnet",
        alpha=alpha,
        l1_ratio=l1_ratio,
        max_iter=40,
        tol=1e-3,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=SEED + fold,
    )
    sgd_lr.fit(X_tr_poly, y_cpu[sub_idx], sample_weight=w_cpu[sub_idx])
    del X_tr_poly

    sk_val_preds = np.zeros(len(val_idx), dtype=np.float32)
    for i in range(0, len(val_idx), 50000):
        c_idx = val_idx[i : i + 50000]
        X_c = np.hstack([
            poly_transformer.transform(X_num_cpu[c_idx][:, top_poly_idx]),
            X_num_cpu[c_idx][:, rem_idx]
        ])
        sk_val_preds[i : i + 50000] = sgd_lr.predict_proba(X_c)[:, 1]

    sk_test_preds = np.zeros(len(X_test_num_cpu), dtype=np.float32)
    for i in range(0, len(X_test_num_cpu), 50000):
        X_c = np.hstack([
            poly_transformer.transform(X_test_num_cpu[i : i + 50000, top_poly_idx]),
            X_test_num_cpu[i : i + 50000, rem_idx]
        ])
        sk_test_preds[i : i + 50000] = sgd_lr.predict_proba(X_c)[:, 1]

    val_blend = 0.65 * best_gpu_val + 0.35 * sk_val_preds
    test_blend = 0.65 * gpu_test_preds + 0.35 * sk_test_preds
    return val_blend, test_blend

# ==============================================================================
# 7. OPTUNA TUNING ON STRICT OUT-OF-TIME SPLIT (TRAIN M1-18 -> VAL M19-24)
# ==============================================================================
def tune_lgb_on_oot_split(X, y, cat_indices, tr_mask, va_mask, n_trials=8):
    print(
        f"\n[{time.time() - T_START:5.1f}s] Tuning LightGBM on Strict OOT Split "
        f"(Train Months 1-18 -> Validate Months 19-24, {n_trials} trials)..."
    )
    tr_indices = np.where(tr_mask)[0]
    va_indices = np.where(va_mask)[0]

    if len(tr_indices) > 350_000:
        rng = np.random.RandomState(SEED)
        tr_sub = rng.choice(tr_indices, size=350_000, replace=False)
    else:
        tr_sub = tr_indices

    base_ds_params = {"max_bin": 255, "feature_pre_filter": False, "verbose": -1}
    dtrain_opt = lgb.Dataset(
        X[tr_sub], label=y[tr_sub], categorical_feature=cat_indices,
        params=base_ds_params, free_raw_data=False
    ).construct()
    dval_opt = lgb.Dataset(
        X[va_indices], label=y[va_indices], reference=dtrain_opt,
        categorical_feature=cat_indices, params=base_ds_params, free_raw_data=False
    ).construct()
    y_va = y[va_indices]

    def objective(trial):
        params = {
            "objective": "binary",
            "metric": "auc",
            "boosting_type": "gbdt",
            "device_type": LGB_DEVICE,
            "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.08, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 31, 127),
            "max_depth": trial.suggest_int("max_depth", 5, 9),
            "min_child_samples": trial.suggest_int("min_child_samples", 50, 300),
            "subsample": trial.suggest_float("subsample", 0.65, 0.90),
            "subsample_freq": 1,
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.50, 0.85),
            "reg_alpha": trial.suggest_float("reg_alpha", 1.0, 20.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 2.0, 30.0, log=True),
            "cat_smooth": trial.suggest_float("cat_smooth", 20.0, 100.0),
            "max_bin": 255,
            "feature_pre_filter": False,
            "n_jobs": -1,
            "seed": SEED,
            "verbose": -1,
        }
        bst = lgb.train(
            params,
            dtrain_opt,
            num_boost_round=800,
            valid_sets=[dval_opt],
            callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(0)],
        )
        preds = bst.predict(X[va_indices], num_iteration=bst.best_iteration)
        return roc_auc_score(y_va, preds)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, timeout=180)
    print(f"[{time.time() - T_START:5.1f}s] OOT Months 19-24 ROC-AUC: {study.best_value:.5f} | Best Params: {study.best_params}")
    del dtrain_opt, dval_opt
    gc.collect()
    return study.best_params

best_lgb_params = tune_lgb_on_oot_split(X_all, y_all, CAT_INDICES, oot_tr_mask, oot_va_mask, n_trials=8)

# ==============================================================================
# 8. EXPANDING WINDOW OOT TRAINING: LIGHTGBM, CATBOOST, TabM, RF, POLY-LR
# ==============================================================================
def train_expanding_oot_models(
    X_tree, y, X_test_tree, cat_indices,
    X_num_dl, X_test_num_dl, X_cat_dl, X_test_cat_dl,
    cat_cards, expanding_folds, ts_rank, best_lgb_params
):
    n_total_train = len(y)
    oof_lgb = np.full(n_total_train, np.nan, dtype=np.float32)
    pred_lgb = np.zeros(len(X_test_tree), dtype=np.float32)

    oof_cat = np.full(n_total_train, np.nan, dtype=np.float32)
    pred_cat = np.zeros(len(X_test_tree), dtype=np.float32)

    oof_tabm = np.full(n_total_train, np.nan, dtype=np.float32)
    pred_tabm = np.zeros(len(X_test_tree), dtype=np.float32)

    oof_rf = np.full(n_total_train, np.nan, dtype=np.float32)
    pred_rf = np.zeros(len(X_test_tree), dtype=np.float32)

    oof_plr = np.full(n_total_train, np.nan, dtype=np.float32)
    pred_plr = np.zeros(len(X_test_tree), dtype=np.float32)

    # Recency-weighted fold weights for test set: give highest weight to Fold 3 (trained up to Month 20)
    fold_test_weights = [0.20, 0.30, 0.50]

    # Sample weights emphasizing more recent transactions inside each expanding window
    sample_weights = (0.75 + 0.75 * ts_rank).astype(np.float32)

    # Select top 14 highest-signal numeric features on Block 0..2 (Months 1-12) for Degree-2 Polynomials
    n_top_poly = min(14, X_num_dl.shape[1])
    m12_idx = expanding_folds[0][0]
    y_cent = y[m12_idx].astype(np.float32) - float(y[m12_idx].mean())
    corrs = np.abs(np.dot(X_num_dl[m12_idx].T, y_cent)) / (len(m12_idx) + 1.0)
    top_poly_idx = np.argsort(corrs)[::-1][:n_top_poly].copy()

    poly_transformer = PolynomialFeatures(degree=2, include_bias=False)
    poly_transformer.fit(X_num_dl[:10, top_poly_idx])
    quad_rows = [r for r in poly_transformer.powers_ if r.sum() == 2]
    q_i = np.array([np.where(r > 0)[0][0] for r in quad_rows], dtype=np.int64)
    q_j = np.array([np.where(r > 0)[0][-1] for r in quad_rows], dtype=np.int64)

    # Stage 1: Train GPU TabM (K=8 Ensemble) & GPU+SGD Polynomial ElasticNet LR on Expanding Windows
    print(
        f"\n[{time.time() - T_START:5.1f}s] Stage 1: Training TabM (GPU K=8) & "
        f"Polynomial ElasticNet LR across 3 Expanding OOT Windows..."
    )
    X_num_all_gpu = torch.from_numpy(X_num_dl).to(DEVICE)
    X_cat_all_gpu = torch.from_numpy(X_cat_dl).to(DEVICE)
    y_all_gpu = torch.from_numpy(y.astype(np.float32)).to(DEVICE)
    w_all_gpu = torch.from_numpy(sample_weights).to(DEVICE)
    X_num_test_gpu = torch.from_numpy(X_test_num_dl).to(DEVICE)
    X_cat_test_gpu = torch.from_numpy(X_test_cat_dl).to(DEVICE)

    for fold, (tr_idx, val_idx, window_name) in enumerate(expanding_folds):
        t_s1 = time.time()
        fw = fold_test_weights[fold]

        val_p_tabm, te_p_tabm = train_tabm_fold_vram(
            X_num_all_gpu, X_cat_all_gpu, y_all_gpu, w_all_gpu,
            X_num_test_gpu, X_cat_test_gpu,
            tr_idx, val_idx, y[val_idx], cat_cards, fold,
            k=8, epochs=12, batch_size=8192, max_lr=2.5e-3
        )
        oof_tabm[val_idx] = val_p_tabm
        pred_tabm += fw * te_p_tabm

        val_p_plr, te_p_plr = train_poly_elasticnet_fold(
            X_num_all_gpu, y_all_gpu, w_all_gpu, X_num_test_gpu,
            X_num_dl, y, sample_weights, X_test_num_dl,
            tr_idx, val_idx, top_poly_idx, q_i, q_j, poly_transformer, fold,
            alpha=1e-3, l1_ratio=0.5, epochs=8, batch_size=16384
        )
        oof_plr[val_idx] = val_p_plr
        pred_plr += fw * te_p_plr

        print(
            f"  OOT Fold {fold+1} [{window_name}] ({time.time() - t_s1:.1f}s) | "
            f"TabM (GPU K=8) AUC: {roc_auc_score(y[val_idx], val_p_tabm):.5f} | "
            f"Poly-ElasticNet LR AUC: {roc_auc_score(y[val_idx], val_p_plr):.5f}"
        )

    del X_num_all_gpu, X_cat_all_gpu, y_all_gpu, w_all_gpu, X_num_test_gpu, X_cat_test_gpu
    if USE_GPU:
        torch.cuda.empty_cache()
    gc.collect()

    # Stage 2: Train LightGBM, CatBoost (GPU), and Random Forest (GPU/LGB-RF) on Expanding Windows
    print(f"\n[{time.time() - T_START:5.1f}s] Stage 2: Training LightGBM, CatBoost (GPU) & Random Forest...")
    test_pool_cat = Pool(X_test_tree)

    lgb_params = {
        **best_lgb_params,
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "device_type": LGB_DEVICE,
        "subsample_freq": 1,
        "max_bin": 255,
        "feature_pre_filter": False,
        "n_jobs": -1,
        "verbose": -1,
    }

    rf_lgb_params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "rf",
        "device_type": LGB_DEVICE,
        "num_leaves": 127,
        "max_depth": 10,
        "min_child_samples": 80,
        "bagging_fraction": 0.80,
        "bagging_freq": 1,
        "feature_fraction": 0.60,
        "reg_lambda": 5.0,
        "cat_smooth": 50.0,
        "max_bin": 255,
        "feature_pre_filter": False,
        "n_jobs": -1,
        "verbose": -1,
    }

    cat_params = {
        "iterations": 1500,
        "learning_rate": 0.06,
        "depth": 7,
        "l2_leaf_reg": 10.0,
        "border_count": 128,
        "eval_metric": "AUC",
        "task_type": "GPU" if USE_GPU else "CPU",
        "early_stopping_rounds": 50,
        "verbose": 0,
    }

    for fold, (tr_idx, val_idx, window_name) in enumerate(expanding_folds):
        f_start = time.time()
        fw = fold_test_weights[fold]
        X_tr, y_tr, w_tr = X_tree[tr_idx], y[tr_idx], sample_weights[tr_idx]
        X_va, y_va = X_tree[val_idx], y[val_idx]

        # 1. LightGBM GBDT
        ds_params = {"max_bin": 255, "feature_pre_filter": False, "verbose": -1}
        dtrain = lgb.Dataset(
            X_tr, label=y_tr, weight=w_tr, categorical_feature=cat_indices,
            params=ds_params, free_raw_data=False
        ).construct()
        dval = lgb.Dataset(
            X_va, label=y_va, reference=dtrain, categorical_feature=cat_indices,
            params=ds_params, free_raw_data=False
        ).construct()

        lgb_params["seed"] = SEED + fold
        bst_lgb = lgb.train(
            lgb_params,
            dtrain,
            num_boost_round=1600,
            valid_sets=[dval],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
        )
        oof_lgb[val_idx] = bst_lgb.predict(X_va, num_iteration=bst_lgb.best_iteration)
        pred_lgb += fw * bst_lgb.predict(X_test_tree, num_iteration=bst_lgb.best_iteration)

        # 2. Random Forest (GPU cuML RF or LightGBM native RF reusing pre-binned dtrain)
        if HAS_CUML:
            m_rf = cuRF(n_estimators=250, max_depth=11, n_bins=128, random_state=SEED + fold)
            m_rf.fit(X_tr, y_tr)
            oof_rf[val_idx] = m_rf.predict_proba(X_va)[:, 1]
            pred_rf += fw * m_rf.predict_proba(X_test_tree)[:, 1]
        else:
            rf_lgb_params["seed"] = SEED + fold
            bst_rf = lgb.train(rf_lgb_params, dtrain, num_boost_round=200)
            oof_rf[val_idx] = bst_rf.predict(X_va)
            pred_rf += fw * bst_rf.predict(X_test_tree)

        del dtrain, dval, bst_lgb
        gc.collect()

        # 3. CatBoost (Native GPU Pool)
        tr_pool = Pool(X_tr, y_tr, weight=w_tr)
        va_pool = Pool(X_va, y_va)
        cat_params["random_seed"] = SEED + fold
        m_cat = CatBoostClassifier(**cat_params)
        m_cat.fit(tr_pool, eval_set=va_pool, verbose=False)
        oof_cat[val_idx] = m_cat.predict_proba(va_pool)[:, 1]
        pred_cat += fw * m_cat.predict_proba(test_pool_cat)[:, 1]
        del tr_pool, va_pool, m_cat
        gc.collect()

        print(
            f"[{time.time() - T_START:5.1f}s] OOT Fold {fold+1} [{window_name}] ({time.time() - f_start:.1f}s) | "
            f"LGB: {roc_auc_score(y_va, oof_lgb[val_idx]):.5f} | "
            f"CAT: {roc_auc_score(y_va, oof_cat[val_idx]):.5f} | "
            f"TabM: {roc_auc_score(y_va, oof_tabm[val_idx]):.5f} | "
            f"RF: {roc_auc_score(y_va, oof_rf[val_idx]):.5f} | "
            f"PolyLR: {roc_auc_score(y_va, oof_plr[val_idx]):.5f}"
        )

    # Evaluate strictly on the Out-of-Time validation windows (Months 13-24: Blocks 3, 4, 5)
    oot_eval_mask = ~np.isnan(oof_lgb)
    y_oot = y[oot_eval_mask]

    print("\n================ OUT-OF-TIME (MONTHS 13-24) VALIDATION SUMMARY ================")
    print(f"OOT LightGBM                       ROC-AUC: {roc_auc_score(y_oot, oof_lgb[oot_eval_mask]):.5f}")
    print(f"OOT CatBoost (GPU)                 ROC-AUC: {roc_auc_score(y_oot, oof_cat[oot_eval_mask]):.5f}")
    print(f"OOT TabM (GPU K=8 DL Ensemble)     ROC-AUC: {roc_auc_score(y_oot, oof_tabm[oot_eval_mask]):.5f}")
    print(f"OOT Random Forest                  ROC-AUC: {roc_auc_score(y_oot, oof_rf[oot_eval_mask]):.5f}")
    print(f"OOT Poly-ElasticNet Logistic Reg   ROC-AUC: {roc_auc_score(y_oot, oof_plr[oot_eval_mask]):.5f}")

    return (
        (oof_lgb, pred_lgb),
        (oof_cat, pred_cat),
        (oof_tabm, pred_tabm),
        (oof_rf, pred_rf),
        (oof_plr, pred_plr),
        oot_eval_mask,
    )

(
    (oof_lgb, pred_lgb),
    (oof_cat, pred_cat),
    (oof_tabm, pred_tabm),
    (oof_rf, pred_rf),
    (oof_plr, pred_plr),
    oot_eval_mask,
) = train_expanding_oot_models(
    X_all, y_all, X_test, CAT_INDICES,
    X_tr_num_dl, X_te_num_dl, X_tr_cat_dl, X_te_cat_dl,
    CAT_CARDS, EXPANDING_FOLDS, tr_ts_rank, best_lgb_params
)

# ==============================================================================
# 9. CLEAN 5-MODEL OOT PROBABILITY BLEND & SUBMISSION GENERATION
# ==============================================================================
W_LGB  = 0.36
W_CAT  = 0.32
W_TABM = 0.17
W_RF   = 0.08
W_PLR  = 0.07

oof_blend = (
    W_LGB  * oof_lgb[oot_eval_mask]
    + W_CAT  * oof_cat[oot_eval_mask]
    + W_TABM * oof_tabm[oot_eval_mask]
    + W_RF   * oof_rf[oot_eval_mask]
    + W_PLR  * oof_plr[oot_eval_mask]
)
test_blend = (
    W_LGB  * pred_lgb
    + W_CAT  * pred_cat
    + W_TABM * pred_tabm
    + W_RF   * pred_rf
    + W_PLR  * pred_plr
)

m21_24_idx = EXPANDING_FOLDS[2][1]
m21_24_blend = (
    W_LGB  * oof_lgb[m21_24_idx]
    + W_CAT  * oof_cat[m21_24_idx]
    + W_TABM * oof_tabm[m21_24_idx]
    + W_RF   * oof_rf[m21_24_idx]
    + W_PLR  * oof_plr[m21_24_idx]
)

print(f"\n>>> Final 5-Model Blend OOT ROC-AUC (Months 13-24 Expanding CV): {roc_auc_score(y_all[oot_eval_mask], oof_blend):.5f} <<<")
print(f">>> Final 5-Model Blend Latest OOT Window (Months 21-24):        {roc_auc_score(y_all[m21_24_idx], m21_24_blend):.5f} <<<")

submission = pd.DataFrame({
    "order_line_id": test_line_ids,
    "returned": np.clip(test_blend, 0.0, 1.0)
})
submission = sub_df[["order_line_id"]].merge(submission, on="order_line_id", how="left")
assert len(submission) == 479995 and submission["returned"].isna().sum() == 0

submission.to_csv("submission11.csv", index=False)
print(f"[{time.time() - T_START:5.1f}s] Saved 'submission11.csv' successfully!")
print(submission["returned"].describe())

c:\python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[  0.1s] PyTorch Device: cuda | LGB Device: cpu | cuML RF: False
[  7.8s] Loaded train: (1400005, 27) | test: (479995, 26)
[  9.2s] Reference tables cleaned & macro/post-holiday features engineered.
[ 29.1s] Feature engineering complete: 160 total features.
[ 64.8s] Matrices ready | Tree: (1400005, 160) | TabM/Poly Num: (1400005, 137)

[ 64.8s] Tuning LightGBM on Strict OOT Split (Train Months 1-18 -> Validate Months 19-24, 8 trials)...
[137.2s] OOT Months 19-24 ROC-AUC: 0.66764 | Best Params: {'learning_rate': 0.033393266539761206, 'num_leaves': 83, 'max_depth': 9, 'min_child_samples': 105, 'subsample': 0.8856717312513483, 'colsample_bytree': 0.5057662263150294, 'reg_alpha': 14.791164122549514, 'reg_lambda': 17.00544144107715, 'cat_smooth': 44.39988187116528}

[137.7s] Stage 1: Training TabM (GPU K=8) & Polynomial ElasticNet LR across 3 Expanding OOT Windows...
  OOT Fold 1 [Months 13-16] (92.5s) | TabM (GPU K=8) AUC: 0.75448 | Poly-ElasticNet LR AUC: 0.69173
  OOT Fold 2 [Months 17-2

Default metric period is 5 because AUC is/are not implemented for GPU


[579.5s] OOT Fold 1 [Months 13-16] (90.4s) | LGB: 0.74143 | CAT: 0.74800 | TabM: 0.75448 | RF: 0.73883 | PolyLR: 0.69173


Default metric period is 5 because AUC is/are not implemented for GPU


[672.7s] OOT Fold 2 [Months 17-20] (92.9s) | LGB: 0.70763 | CAT: 0.71812 | TabM: 0.70125 | RF: 0.70375 | PolyLR: 0.65894


Default metric period is 5 because AUC is/are not implemented for GPU


[899.3s] OOT Fold 3 [Months 21-24] (226.3s) | LGB: 0.68931 | CAT: 0.70872 | TabM: 0.72418 | RF: 0.66446 | PolyLR: 0.67782

================ OUT-OF-TIME (MONTHS 13-24) VALIDATION SUMMARY ================
OOT LightGBM                       ROC-AUC: 0.71270
OOT CatBoost (GPU)                 ROC-AUC: 0.72547
OOT TabM (GPU K=8 DL Ensemble)     ROC-AUC: 0.72584
OOT Random Forest                  ROC-AUC: 0.70378
OOT Poly-ElasticNet Logistic Reg   ROC-AUC: 0.67479

>>> Final 5-Model Blend OOT ROC-AUC (Months 13-24 Expanding CV): 0.72518 <<<
>>> Final 5-Model Blend Latest OOT Window (Months 21-24):        0.71194 <<<
[902.0s] Saved 'submission11.csv' successfully!
count    479995.000000
mean          0.194758
std           0.092295
min           0.028996
25%           0.130587
50%           0.172540
75%           0.235211
max           0.814932
Name: returned, dtype: float64
